In [1]:
from pyspark.sql import functions as F

# 1. Load and Force Root Integrity
table_name = "support_rls_dynamic_merged"
df_raw = spark.read.table(table_name)

# DATA FIX: Force 4037 to root
df = df_raw.select(
    F.col("PersonFk").cast("string"),
    F.trim(F.col("SupervisorId")).cast("string").alias("SupervisorId")
).withColumn(
    "SupervisorId",
    F.when(F.col("PersonFk") == "4037", None) # Absolute Root
     .when(F.col("SupervisorId") == "", None)
     .otherwise(F.col("SupervisorId"))
)

# 2. Initial State
hierarchy_df = df.withColumn("OrgPath", F.col("PersonFk")) \
                 .withColumn("CurrentHead", F.col("SupervisorId"))

# 3. Iterative Build (Safe against any other potential loops)
max_depth = 15 

for i in range(max_depth):
    temp_df = hierarchy_df.alias("c") \
        .join(df.alias("p"), F.col("c.CurrentHead") == F.col("p.PersonFk"), "left") \
        .select(
            F.col("c.*"),
            F.col("p.SupervisorId").alias("NextParent")
        )
    
    hierarchy_df = temp_df.withColumn(
        "OrgPath",
        F.when(
            F.col("CurrentHead").isNotNull() & 
            ~F.col("OrgPath").contains(F.col("CurrentHead")), 
            F.concat(F.col("CurrentHead"), F.lit("|"), F.col("OrgPath"))
        ).otherwise(F.col("OrgPath"))
    ).withColumn(
        "CurrentHead", 
        F.when(F.col("OrgPath").contains(F.col("NextParent")), None) 
         .otherwise(F.col("NextParent"))
    ).drop("NextParent")

# 4. Final Cleanup
final_hierarchy = hierarchy_df.drop("CurrentHead", "SupervisorId")
final_df = final_hierarchy.join(df_raw, "PersonFk", "inner")

# 5. Write to Gold
final_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("support_rls_hierarchy_gold")

StatementMeta(, e9d4ee5b-a400-4005-899a-1abbfc59a0e0, 3, Finished, Available, Finished, False)